<a href="https://colab.research.google.com/github/vad-source/NLPAPP/blob/main/SENTIMENT/NLPAPP_LexiconBased_SA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## NLP APPLICATIONS
**Designed by:** RAJA VADHANA PRABHAKAR  
**Organization:** BITS PILANI WILP  
**Purpose:** Academic Training / Proof of Concept  

---
#### Attribution & AI Disclosure
- **Original Design:** The logic, architecture, and modular structure of this notebook were designed by the author.
- **Development Assistance:** Generative AI (e.g., ChatGPT/Claude/Copilot) was used for coding implementation and debugging support.
- **License:** This work is licensed under the [Apache License 2.0](https://apache.org).

In [ ]:
!pip install nltk spacy pandas networkx -q

In [ ]:
import re
import json
import uuid
import hashlib
import unicodedata
import pandas as pd
import networkx as nx

import nltk

In [ ]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('sentiwordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package sentiwordnet to /root/nltk_data...
[nltk_data]   Unzipping corpora/sentiwordnet.zip.


True

In [ ]:
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [ ]:
from nltk.corpus import sentiwordnet as swn
from nltk.corpus import wordnet as wn
from nltk import pos_tag, word_tokenize

import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except:
    !python -m spacy download en_core_web_sm
    nlp = spacy.load("en_core_web_sm")

## 0 : Ingestion (source Data)

In [ ]:
from abc import ABC, abstractmethod
class PipelineComponent(ABC):
    @abstractmethod
    def run(self, payload):
        pass
class InferenceEngine(ABC):
    @abstractmethod
    def predict(self, payload):
        pass

## 1 : Preprocessor

### 1.1 : Inbound Guardrails

In [ ]:
class InboundGuardrails(PipelineComponent):
    MAX_LEN = 1000
    BLOCK_PATTERNS = [
        r"<script",
        r"DROP TABLE",
        r"DELETE FROM",
        r"IGNORE PREVIOUS INSTRUCTIONS"
    ]
    def run(self, text):
        if not text:
            raise ValueError("Empty Payload")
        if len(text) > self.MAX_LEN:
            raise ValueError("Payload Too Large")
        for pattern in self.BLOCK_PATTERNS:
            if re.search(pattern, text, re.IGNORECASE):
                raise ValueError(f"Blocked Pattern: {pattern}")
        return text

In [ ]:
text = "Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: XYZ@abc.com 1234567899. Use sk-ABC123XYZ"
guardT = InboundGuardrails()
print(guardT.run(text))

Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: XYZ@abc.com 1234567899. Use sk-ABC123XYZ


In [ ]:
class SentimentPlatform:
    def __init__(self, engine):
        self.guard = InboundGuardrails()
        self.compliance = CompliancePII()
        self.normalize = TextNormalizer()
        self.vectorize = DocumentVectorizer()
        self.engine = engine
        self.telemetry = (ExplainabilityTelemetry())
        self.outbound = (OutboundGuardrails())
    def analyze(self, text):
        text = self.guard.run(text)
        text = self.compliance.run(text)
        text = self.normalize.run(text)
        vectors = self.vectorize.run(text)
        result = self.engine.predict(vectors)
        result = self.telemetry.run(result)
        result = self.outbound.run(result)
        return result

### 1.2 : Compliance Check

In [ ]:
class CompliancePII(PipelineComponent):
    EMAIL = r'\S+@\S+'
    PHONE = r'\d{10}'
    APIKEY = r'sk-[A-Za-z0-9]+'
    def run(self, text):
        text = re.sub(self.EMAIL,'[EMAIL]',text)
        text = re.sub(self.PHONE,'[PHONE]',text)
        text = re.sub(self.APIKEY,'[APIKEY]',text)
        return text

In [ ]:
text = "Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: XYZ@abc.com 1234567899. Use sk-ABC123XYZ"
complianceT = CompliancePII()
print(complianceT.run(guardT.run(text)))

Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: [EMAIL] [PHONE]. Use [APIKEY]


### 1.3 : Normalization

In [ ]:
class TextNormalizer(PipelineComponent):
    def run(self, text):
        text = unicodedata.normalize("NFKC",text)
        text = text.lower()
        text = re.sub(r"\s+"," ",text)
        return text.strip()

In [ ]:
text = "Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: XYZ@abc.com 1234567899. Use sk-ABC123XYZ"
normalizeT = TextNormalizer()
print(normalizeT.run(complianceT.run(guardT.run(text))))

can’t even comment. customer service was not very terrible !!!! . contact me: [email] [phone]. use [apikey]


## 2 : Representation & Processing

In [ ]:
class DocumentVectorizer(PipelineComponent):
    def run(self, text):
        return {"raw_text": text,"tokens": word_tokenize(text),"pos_tags": pos_tag(word_tokenize(text)),"spacy_doc": nlp(text)}

In [ ]:
text = "Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: XYZ@abc.com 1234567899. Use sk-ABC123XYZ"
vectorizeT = DocumentVectorizer()
print(vectorizeT.run(normalizeT.run(complianceT.run(guardT.run(text)))))

{'raw_text': 'can’t even comment. customer service was not very terrible !!!! . contact me: [email] [phone]. use [apikey]', 'tokens': ['can', '’', 't', 'even', 'comment', '.', 'customer', 'service', 'was', 'not', 'very', 'terrible', '!', '!', '!', '!', '.', 'contact', 'me', ':', '[', 'email', ']', '[', 'phone', ']', '.', 'use', '[', 'apikey', ']'], 'pos_tags': [('can', 'MD'), ('’', 'VB'), ('t', 'VB'), ('even', 'RB'), ('comment', 'NN'), ('.', '.'), ('customer', 'NN'), ('service', 'NN'), ('was', 'VBD'), ('not', 'RB'), ('very', 'RB'), ('terrible', 'JJ'), ('!', '.'), ('!', '.'), ('!', '.'), ('!', '.'), ('.', '.'), ('contact', 'VB'), ('me', 'PRP'), (':', ':'), ('[', 'JJ'), ('email', 'NN'), (']', 'NNP'), ('[', 'NNP'), ('phone', 'NN'), (']', 'NN'), ('.', '.'), ('use', 'NN'), ('[', 'JJ'), ('apikey', 'NN'), (']', 'NN')], 'spacy_doc': can’t even comment. customer service was not very terrible !!!! . contact me: [email] [phone]. use [apikey]}


## 3 : Analyser (Inference Engine)

### A : Document Level

In [ ]:
class DocumentLevelLexiconEngine(InferenceEngine):
    def predict(self, payload):
        sentences = re.split( r'(?<=[.!?])\s+',payload["raw_text"])
        sentence_scores = []
        explanations = []
        for sentence in sentences:
            score = 0
            tagged = pos_tag(word_tokenize(sentence))
            for word, tag in tagged:
                if tag.startswith('J'):
                    pos = wn.ADJ
                elif tag.startswith('N'):
                    pos = wn.NOUN
                elif tag.startswith('V'):
                    pos = wn.VERB
                else:
                    pos = None

                if pos:
                    synsets = wn.synsets(word,pos=pos)
                    if synsets:
                        try:
                            senti = (swn.senti_synset(synsets[0].name()))
                            contribution = (senti.pos_score()- senti.neg_score())
                            score += contribution
                            explanations.append({"token": word,"impact": contribution})
                        except:
                            pass
            sentence_scores.append(score)
        final_score = (sum(sentence_scores)/ max(1, len(sentence_scores)))
        return {"task":"document","score":final_score,
                "sentiment":"POSITIVE"
                if final_score>0
                else "NEGATIVE",
                "explanations":explanations}

In [ ]:
text = "Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: XYZ@abc.com 1234567899. Use sk-ABC123XYZ"
engineDT = DocumentLevelLexiconEngine()
print(engineDT.predict(vectorizeT.run(normalizeT.run(complianceT.run(guardT.run(text))))))

{'task': 'document', 'score': -0.025, 'sentiment': 'NEGATIVE', 'explanations': [{'token': 'comment', 'impact': 0.0}, {'token': 'customer', 'impact': 0.0}, {'token': 'service', 'impact': 0.0}, {'token': 'was', 'impact': 0.125}, {'token': 'terrible', 'impact': -0.625}, {'token': 'contact', 'impact': 0.375}, {'token': 'email', 'impact': 0.0}, {'token': 'phone', 'impact': 0.0}, {'token': 'use', 'impact': 0.0}]}


#### B : Sentence Level

In [ ]:
NEGATIONS = {"not","never","no"}
INTENSIFIERS = {"very":1.5,"extremely":2.0,"really":1.4}
BASE_LEXICON = {"good":2,"great":3,"excellent":4,"bad":-2,"terrible":-4,"awful":-4,"happy":2,"angry":-3}

In [ ]:
class SentenceLevelValenceEngine(InferenceEngine):
    def predict(self, payload):
        tokens = payload["tokens"]
        score = 0
        explain = []
        for i, token in enumerate(tokens):
            word = token.lower()
            if word not in BASE_LEXICON:
                continue
            current = BASE_LEXICON[word]
            context = tokens[max(0, i-3):i]
            for c in context:
                c = c.lower()
                if c in NEGATIONS:
                    current *= -1
                if c in INTENSIFIERS:
                    current *= INTENSIFIERS[c]
            explain.append({"token":word,"impact":current})
            score += current
        return {"task":"sentence","score":score,"sentiment":"POSITIVE"if score>0 else "NEGATIVE","explanations":explain}

In [ ]:
text = "Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: XYZ@abc.com 1234567899. Use sk-ABC123XYZ"
engineST = SentenceLevelValenceEngine()
print(engineST.predict(vectorizeT.run(normalizeT.run(complianceT.run(guardT.run(text))))))

{'task': 'sentence', 'score': 6.0, 'sentiment': 'POSITIVE', 'explanations': [{'token': 'terrible', 'impact': 6.0}]}


#### C : Aspect Level

In [ ]:
class AspectLevelDependencyEngine(InferenceEngine):
    def predict(self, payload):
        doc = payload["spacy_doc"]
        graph = nx.Graph()
        for token in doc:
            graph.add_node(token.i)
            if token.head.i != token.i:
                graph.add_edge(token.i,token.head.i)
        nouns = [t for t in doc if t.pos_=="NOUN"]
        adjectives = [t for t in doc if t.pos_=="ADJ"]
        aspect_scores = []
        for noun in nouns:
            score = 0
            contributors = []
            for adj in adjectives:
                try:
                    distance = nx.shortest_path_length(graph,noun.i,adj.i)
                    synsets = wn.synsets(adj.text,pos=wn.ADJ)
                    if synsets:
                        senti = swn.senti_synset(synsets[0].name())
                        polarity = (senti.pos_score()- senti.neg_score())
                        contribution = (polarity/(distance+1))
                        score += contribution
                        contributors.append({"adj":adj.text,"distance":distance,"impact":contribution})
                except:
                    pass
            aspect_scores.append({"aspect":noun.text,"score":score,"contributors":contributors})
        return {"task":"aspect","aspects":aspect_scores}

In [ ]:
import spacy
text = "Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: XYZ@abc.com 1234567899. Use sk-ABC123XYZ"
nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
spacy.displacy.serve(doc, style="dep")

/usr/local/lib/python3.12/dist-packages/spacy/displacy/__init__.py:107: UserWarning: [W011] It looks like you're calling displacy.serve from within a Jupyter notebook or a similar environment. This likely means you're already running a local web server, so there's no need to make displaCy start another one. Instead, you should be able to replace displacy.serve with displacy.render to show the visualization.
  warnings.warn(Warnings.W011)



Using the 'dep' visualizer
Serving on http://0.0.0.0:5000 ...

Shutting down server on port 5000.


In [ ]:
text = "Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: XYZ@abc.com 1234567899. Use sk-ABC123XYZ"
engineAT = AspectLevelDependencyEngine()
print(engineAT.predict(vectorizeT.run(normalizeT.run(complianceT.run(guardT.run(text))))))

{'task': 'aspect', 'aspects': [{'aspect': 'customer', 'score': -0.15625, 'contributors': [{'adj': 'terrible', 'distance': 3, 'impact': -0.15625}]}, {'aspect': 'service', 'score': -0.20833333333333334, 'contributors': [{'adj': 'terrible', 'distance': 2, 'impact': -0.20833333333333334}]}, {'aspect': 'email', 'score': 0, 'contributors': []}, {'aspect': 'phone', 'score': 0, 'contributors': []}]}


## 4 : Post Processor

### 4.1 Explainer

In [ ]:
class ExplainabilityTelemetry(PipelineComponent):
    def run(self, inference):
        inference["telemetry"] = {"trace_id":str(uuid.uuid4()),"model_version":"rule_based_v1","feature_count":len(inference.get("explanations",[]))}
        return inference

In [ ]:
text = "Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: XYZ@abc.com 1234567899. Use sk-ABC123XYZ"
telemetryT = (ExplainabilityTelemetry())
print(telemetryT.run(engineAT.predict(vectorizeT.run(normalizeT.run(complianceT.run(guardT.run(text)))))))

{'task': 'aspect', 'aspects': [{'aspect': 'customer', 'score': -0.15625, 'contributors': [{'adj': 'terrible', 'distance': 3, 'impact': -0.15625}]}, {'aspect': 'service', 'score': -0.20833333333333334, 'contributors': [{'adj': 'terrible', 'distance': 2, 'impact': -0.20833333333333334}]}, {'aspect': 'email', 'score': 0, 'contributors': []}, {'aspect': 'phone', 'score': 0, 'contributors': []}], 'telemetry': {'trace_id': '3472b57e-f0bc-4a31-9f5b-3fd10a6e19ba', 'model_version': 'rule_based_v1', 'feature_count': 0}}


### 4.2 : Outbound Guardrails

In [ ]:
class OutboundGuardrails(PipelineComponent):
    def run(self, result):
        if "score" in result:
            score = abs(result["score"])
            result["confidence"] = min(1.0,score/5)
        return result

In [ ]:
outboundT = (OutboundGuardrails())
text = "Can’t even comment. Customer service was not very TERRIBLE !!!! . Contact me: XYZ@abc.com 1234567899. Use sk-ABC123XYZ"
print(outboundT.run(telemetryT.run(engineAT.predict(vectorizeT.run(normalizeT.run(complianceT.run(guardT.run(text))))))))

{'task': 'aspect', 'aspects': [{'aspect': 'customer', 'score': -0.15625, 'contributors': [{'adj': 'terrible', 'distance': 3, 'impact': -0.15625}]}, {'aspect': 'service', 'score': -0.20833333333333334, 'contributors': [{'adj': 'terrible', 'distance': 2, 'impact': -0.20833333333333334}]}, {'aspect': 'email', 'score': 0, 'contributors': []}, {'aspect': 'phone', 'score': 0, 'contributors': []}], 'telemetry': {'trace_id': '856aa6a1-540a-4783-af4f-254b360722d1', 'model_version': 'rule_based_v1', 'feature_count': 0}}


## Evaluator

## Pipeline

In [ ]:
class SentimentPlatform:
    def __init__(self, engine):
        self.guard = InboundGuardrails()
        self.compliance = CompliancePII()
        self.normalize = TextNormalizer()
        self.vectorize = DocumentVectorizer()
        self.engine = engine
        self.telemetry = (ExplainabilityTelemetry())
        self.outbound = (OutboundGuardrails())
    def analyze(self, text):
        text = self.guard.run(text)
        text = self.compliance.run(text)
        text = self.normalize.run(text)
        vectors = self.vectorize.run(text)
        result = self.engine.predict(vectors)
        result = self.telemetry.run(result)
        result = self.outbound.run(result)
        return result

In [ ]:
platform = SentimentPlatform(DocumentLevelLexiconEngine())
platform.analyze("""Customer support was excellent.Delivery was terrible.Contact me at XYZ@abc.com""")

{'task': 'document',
 'score': 0.25,
 'sentiment': 'POSITIVE',
 'explanations': [{'token': 'customer', 'impact': 0.0},
  {'token': 'support', 'impact': 0.0},
  {'token': 'was', 'impact': 0.125},
  {'token': 'was', 'impact': 0.125},
  {'token': 'email', 'impact': 0.0}],
 'telemetry': {'trace_id': '29266255-c84d-43dc-9c7e-2afd86cc45f0',
  'model_version': 'rule_based_v1',
  'feature_count': 5},
 'confidence': 0.05}

In [ ]:
platform = SentimentPlatform(SentenceLevelValenceEngine())
platform.analyze("I am not very happy with this awful service")

{'task': 'sentence',
 'score': -7.0,
 'sentiment': 'NEGATIVE',
 'explanations': [{'token': 'happy', 'impact': -3.0},
  {'token': 'awful', 'impact': -4}],
 'telemetry': {'trace_id': 'c423f637-4e7f-44dc-ae64-f9bd6e9b9697',
  'model_version': 'rule_based_v1',
  'feature_count': 2},
 'confidence': 1.0}

In [ ]:
platform = SentimentPlatform(AspectLevelDependencyEngine())
platform.analyze("""Food was excellent.Service was terrible.Ambience was beautiful.""")

{'task': 'aspect',
 'aspects': [{'aspect': 'food',
   'score': 0.1875,
   'contributors': [{'adj': 'beautiful', 'distance': 3, 'impact': 0.1875}]},
  {'aspect': 'terrible.ambience',
   'score': 0.25,
   'contributors': [{'adj': 'beautiful', 'distance': 2, 'impact': 0.25}]}],
 'telemetry': {'trace_id': '906cf053-ca6a-419e-bd36-e6b364b35c65',
  'model_version': 'rule_based_v1',
  'feature_count': 0}}